# Stage 6: Walk-Forward LSTM Training

This notebook trains a pooled cross-sectional LSTM in an expanding yearly walk-forward setup and writes strictly out-of-sample predictions to `data/lstm_predictions.csv`.

In [6]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

# Resolve team_t root regardless of launch directory.
CWD = Path.cwd().resolve()
if (CWD / "team_t").exists():
    TEAM_ROOT = CWD / "team_t"
elif CWD.name == "team_t":
    TEAM_ROOT = CWD
elif CWD.name == "notebooks" and CWD.parent.name == "team_t":
    TEAM_ROOT = CWD.parent
else:
    TEAM_ROOT = next((p for p in [CWD, *CWD.parents] if p.name == "team_t"), None)

if TEAM_ROOT is None:
    raise FileNotFoundError("Could not resolve team_t directory from current working directory")

if str(TEAM_ROOT) not in sys.path:
    sys.path.insert(0, str(TEAM_ROOT))

from src.model_utils import (
    build_sequence_dataset,
    load_sequence_dataset_npz,
    save_sequence_dataset_npz,
    walk_forward_lstm_predictions,
)

## Configuration

Defaults follow the requested walk-forward logic: `2006–2009 -> 2010`, then expand one year at a time until the latest available year with sufficient samples.

In [7]:
LOOKBACK = 40
TRAIN_START_YEAR = 2006
FIRST_PREDICT_YEAR = 2010
MAX_PREDICT_YEAR = None  # Set int year to cap prediction horizon.

MIN_TRAIN_SAMPLES = 200
MIN_PREDICT_SAMPLES = 1

HIDDEN_SIZE = 64
DROPOUT = 0.2
LEARNING_RATE = 1e-3
EPOCHS = 8
BATCH_SIZE = 256
RANDOM_SEED = 42
DEVICE = None  # None -> auto-select cuda if available else cpu.

OUTPUT_DIR = TEAM_ROOT / "data" / "lstm_draft" / "processed"
MODEL_PANEL_PARQUET = OUTPUT_DIR / "lstm_model_panel_full_history.parquet"
SEQUENCE_DATASET_NPZ = OUTPUT_DIR / f"lstm_sequence_dataset_lookback_{LOOKBACK}.npz"
WALK_SUMMARY_CSV = OUTPUT_DIR / f"lstm_walk_forward_training_summary_lookback_{LOOKBACK}.csv"
PREDICTIONS_CSV = TEAM_ROOT / "data" / "lstm_predictions.csv"

print(f"TEAM_ROOT={TEAM_ROOT}")
print(f"MODEL_PANEL_PARQUET={MODEL_PANEL_PARQUET}")
print(f"SEQUENCE_DATASET_NPZ={SEQUENCE_DATASET_NPZ}")
print(f"PREDICTIONS_CSV={PREDICTIONS_CSV}")


TEAM_ROOT=/Users/assortedsphinx/Desktop/team_t
MODEL_PANEL_PARQUET=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/lstm_model_panel_full_history.parquet
SEQUENCE_DATASET_NPZ=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/lstm_sequence_dataset_lookback_60.npz
PREDICTIONS_CSV=/Users/assortedsphinx/Desktop/team_t/data/lstm_predictions.csv


## Load or Build Stage-5 Sequence Dataset

The split rule uses `date = sequence end date t`.
Sequences are built within ticker history and are allowed to use prior-year lookback rows for January predictions.

In [8]:
if not MODEL_PANEL_PARQUET.exists():
    raise FileNotFoundError(f"Model panel not found: {MODEL_PANEL_PARQUET}")

panel_model = pd.read_parquet(MODEL_PANEL_PARQUET)
panel_model["date"] = pd.to_datetime(panel_model["date"], errors="coerce")
panel_model = panel_model.dropna(subset=["date", "ticker"]).copy()
panel_model["ticker"] = panel_model["ticker"].astype(str).str.upper().str.strip()
panel_model = panel_model.sort_values(["ticker", "date"]).reset_index(drop=True)

feature_cols = [
    c for c in panel_model.columns
    if c not in {"date", "ticker", "target_return"}
]
assert feature_cols, "No feature columns found in model panel."

needs_rebuild = False
if SEQUENCE_DATASET_NPZ.exists():
    sequence_data = load_sequence_dataset_npz(SEQUENCE_DATASET_NPZ)
    print(f"[cache hit] sequence dataset={SEQUENCE_DATASET_NPZ}")

    assert sequence_data["lookback"] == LOOKBACK, f"Cached lookback {sequence_data['lookback']} != notebook LOOKBACK {LOOKBACK}"
    assert set(sequence_data["feature_cols"]) == set(feature_cols), "Cached sequence dataset features do not match current feature columns."

    if ("label_dates" not in sequence_data) or (sequence_data["label_dates"] is None):
        print("[cache stale] missing label_dates; rebuilding sequence dataset for strict year-boundary split")
        needs_rebuild = True
else:
    needs_rebuild = True

if needs_rebuild:
    sequence_data = build_sequence_dataset(
        panel_df=panel_model,
        feature_cols=feature_cols,
        target_col="target_return",
        lookback=LOOKBACK,
        date_col="date",
        ticker_col="ticker",
    )
    save_sequence_dataset_npz(sequence_data, SEQUENCE_DATASET_NPZ)
    print(f"[export] sequence dataset={SEQUENCE_DATASET_NPZ}")

import numpy as np

assert sequence_data["lookback"] == LOOKBACK, f"Sequence lookback {sequence_data['lookback']} != notebook LOOKBACK {LOOKBACK}"
assert set(sequence_data["feature_cols"]) == set(feature_cols), "Sequence dataset features do not match current feature columns."
assert "label_dates" in sequence_data and sequence_data["label_dates"] is not None, "Sequence dataset missing label_dates; rebuild required."

assert np.isfinite(sequence_data["X_sequences"]).all(), "Sequence dataset contains non-finite feature values"
assert np.isfinite(sequence_data["y_targets"]).all(), "Sequence dataset contains non-finite target values"

sample_dates = pd.to_datetime(sequence_data["sample_dates"], errors="coerce")
label_dates = pd.to_datetime(sequence_data["label_dates"], errors="coerce")

assert len(label_dates) == len(sample_dates), "label_dates length must match sample_dates length"
assert (label_dates > sample_dates).all(), "Each label_date must be strictly after sample_date"

print("sequence samples:", len(sequence_data["X_sequences"]))
print("sequence tensor shape:", sequence_data["X_sequences"].shape)
print("feature count:", len(sequence_data["feature_cols"]))
print("unique tickers:", pd.Series(sequence_data["tickers"]).nunique())
print(f"[sequence dataset] sample_date_min={sample_dates.min()} sample_date_max={sample_dates.max()}")
print(f"[sequence dataset] label_date_min={label_dates.min()} label_date_max={label_dates.max()}")



[cache hit] sequence dataset=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/lstm_sequence_dataset_lookback_60.npz
sequence samples: 41654
sequence tensor shape: (41654, 60, 8)
feature count: 8
unique tickers: 10
[sequence dataset] sample_date_min=2006-06-13 00:00:00 sample_date_max=2024-11-01 00:00:00
[sequence dataset] label_date_min=2006-06-14 00:00:00 label_date_max=2024-11-04 00:00:00


## Expanding Walk-Forward Training and Prediction

For each prediction year `Y`:
- training subset: `sample_date <= Dec 31 of Y-1` and `sample_date >= Jan 1, 2006`
- prediction subset: `Jan 1 of Y` to `Dec 31 of Y`

Each year trains a fresh model from scratch with no shuffling and no temporal overlap between train and prediction subsets.

In [9]:
s = pd.to_datetime(sequence_data["sample_dates"], errors="coerce")
l = pd.to_datetime(sequence_data["label_dates"], errors="coerce")
print(f"[split audit] sample_date_min={s.min()} sample_date_max={s.max()}")
print(f"[split audit] label_date_min={l.min()} label_date_max={l.max()}")
assert (l > s).all(), "label_date must be strictly greater than sample_date for every sequence"

predictions_df, walk_summary_df = walk_forward_lstm_predictions(
    X_sequences=sequence_data["X_sequences"],
    y_targets=sequence_data["y_targets"],
    sample_dates=sequence_data["sample_dates"],
    tickers=sequence_data["tickers"],
    label_dates=sequence_data["label_dates"],
    train_start_year=TRAIN_START_YEAR,
    first_predict_year=FIRST_PREDICT_YEAR,
    max_predict_year=MAX_PREDICT_YEAR,
    min_train_samples=MIN_TRAIN_SAMPLES,
    min_predict_samples=MIN_PREDICT_SAMPLES,
    hidden_size=HIDDEN_SIZE,
    dropout=DROPOUT,
    learning_rate=LEARNING_RATE,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    seed=RANDOM_SEED,
    device=DEVICE,
)

print(f"[walk-forward] trained_years={(walk_summary_df['status'] == 'trained').sum()}")
print(f"[walk-forward] prediction_rows={len(predictions_df):,}")

walk_diag = walk_summary_df.copy()
if "prediction_year" in walk_diag.columns and "predict_year" not in walk_diag.columns:
    walk_diag = walk_diag.rename(columns={"prediction_year": "predict_year"})

print(walk_diag[[
    "predict_year",
    "train_samples",
    "predict_samples",
    "status",
]])



[split audit] sample_date_min=2006-06-13 00:00:00 sample_date_max=2024-11-01 00:00:00
[split audit] label_date_min=2006-06-14 00:00:00 label_date_max=2024-11-04 00:00:00
[walk-forward] train=2006-2009 predict_year=2010 train_samples=5,554 predict_samples=2,016
[walk-forward] train=2006-2010 predict_year=2011 train_samples=7,570 predict_samples=2,240
[walk-forward] train=2006-2011 predict_year=2012 train_samples=9,810 predict_samples=2,250
[walk-forward] train=2006-2012 predict_year=2013 train_samples=12,060 predict_samples=2,304
[walk-forward] train=2006-2013 predict_year=2014 train_samples=14,364 predict_samples=2,520
[walk-forward] train=2006-2014 predict_year=2015 train_samples=16,884 predict_samples=2,520
[walk-forward] train=2006-2015 predict_year=2016 train_samples=19,404 predict_samples=2,520
[walk-forward] train=2006-2016 predict_year=2017 train_samples=21,924 predict_samples=2,510
[walk-forward] train=2006-2017 predict_year=2018 train_samples=24,434 predict_samples=2,510
[walk

## Export Predictions

Required output schema:
`date, ticker, prediction, actual_return`

In [10]:
if predictions_df.empty:
    raise ValueError("Walk-forward predictions are empty. Check data sufficiency thresholds.")

assert np.isfinite(predictions_df["prediction"]).all(),     "Non-finite predictions detected"

assert np.isfinite(predictions_df["actual_return"]).all(),     "Non-finite returns detected"

predictions_df = predictions_df[["date", "ticker", "prediction", "actual_return"]].copy()
predictions_df = predictions_df.sort_values(["date", "ticker"]).reset_index(drop=True)

expected_cols = ["date", "ticker", "prediction", "actual_return"]
assert list(predictions_df.columns) == expected_cols

PREDICTIONS_CSV.parent.mkdir(parents=True, exist_ok=True)
predictions_df.to_csv(PREDICTIONS_CSV, index=False)
walk_summary_df.to_csv(WALK_SUMMARY_CSV, index=False)

print(f"[export] predictions={PREDICTIONS_CSV}")
print(f"[export] walk_summary={WALK_SUMMARY_CSV}")
print(f"[predictions] rows={len(predictions_df):,} tickers={predictions_df['ticker'].nunique():,}")
print(f"[predictions] date_min={predictions_df['date'].min()} date_max={predictions_df['date'].max()}")
predictions_df.head()


[export] predictions=/Users/assortedsphinx/Desktop/team_t/data/lstm_predictions.csv
[export] walk_summary=/Users/assortedsphinx/Desktop/team_t/data/lstm_draft/processed/lstm_walk_forward_training_summary_lookback_60.csv
[predictions] rows=36,100 tickers=10
[predictions] date_min=2009-12-31 00:00:00 date_max=2024-11-01 00:00:00


,date,ticker,prediction,actual_return
0,2009-12-31,BRK_B,-0.000036,0.007579
1,2009-12-31,CVX,0.000459,0.026532
2,2009-12-31,GE,0.000467,0.020929
3,2009-12-31,IBM,0.000899,0.011772
4,2009-12-31,JNJ,0.001413,0.004183
